# Milestone 2 – chạy thử phần Member 1 (gatherer, hierarchy, pipeline) trên dữ liệu thật

Notebook này **chỉ đọc** dữ liệu Milestone 1 trên Drive: nó chép một số bài về ổ của máy Colab (`/content/m2/data`),
rồi chạy trên bản chép đó. Không gửi request nào ra ngoài (ngoài `git clone`).

1. **Bước 1** điền đường dẫn tới dữ liệu của một khoảng: thư mục bản lưu mới (`chinh_thuc/` hoặc `03_benchmark/<tên>_<khoảng>`, có `snapshots/*.tar.gz`)
   hoặc thư mục cũ trên Drive (`02_trial/<tên>_<khoảng>`, có `corpus/` và `work/`).
2. **Bước 3** quét `gatherer.py` trên mọi phiên bản đã chép: tìm file gốc, chuỗi `\input`, file thiếu, bài chỉ có `.bbl`...
3. **Bước 4** chạy `pipeline_m2.py` (phần của Member 2/3/4 chưa có thì dùng bản tối giản trong `m2_contract.py`).
4. **Bước 5** in cây của một bài để nhìn bằng mắt.

*Runtime → Change runtime type → CPU.* Chạy lần lượt từ trên xuống.

In [ ]:
# ===================== ĐIỀN =====================
NGUON   = "/content/drive/MyDrive/group11-data/02_trial/AQuan_2605-12338_2605-17337"  # dữ liệu Milestone 1 của một khoảng
SO_BAI  = 300        # số bài chép về để chạy thử (None = tất cả; thư mục cũ trên Drive chép chậm, cỡ vài bài mỗi giây)
REPO_URL = "https://github.com/mituleefam/group11-arxiv-project.git"
BRANCH   = "quan-m2-member1"
# ================================================
DATA = "/content/m2/data"

## Bước 2. Drive và code

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
os.chdir("/content")
if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q -B {BRANCH} FETCH_HEAD
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r src/requirements.txt

## Bước 2b. Chép dữ liệu về máy Colab
Có bản lưu `.tar.gz` thì khôi phục (nhanh). Thư mục cũ trên Drive thì chép từng bài, nhiều luồng cùng lúc.

In [ ]:
import glob, json, os, shutil, time
from concurrent.futures import ThreadPoolExecutor
os.makedirs(DATA, exist_ok=True)
t0 = time.time()
if glob.glob(f"{NGUON}/snapshots/*.tar.gz"):
    !python src/snapshot.py restore --data-dir "{DATA}" --backup-dir "{NGUON}/snapshots"
else:
    os.makedirs(f"{DATA}/work", exist_ok=True)
    lines = [l for l in open(f"{NGUON}/work/paper_manifest.jsonl", encoding="utf-8") if l.strip()]
    ids = [json.loads(l)["arxiv_id"] for l in lines]
    ids = [i for i in dict.fromkeys(ids) if os.path.isdir(f"{NGUON}/corpus/{i.replace('.', '-')}")][:SO_BAI]
    def chep(arxiv_id):
        src, dst = f"{NGUON}/corpus/{arxiv_id.replace('.', '-')}", f"{DATA}/corpus/{arxiv_id.replace('.', '-')}"
        if not os.path.isdir(dst):
            shutil.copytree(src, dst, ignore=shutil.ignore_patterns("parsed", "*.tmp"))
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(chep, ids))
    chon = set(ids)  # the manifest of the copied papers only
    with open(f"{DATA}/work/paper_manifest.jsonl", "w", encoding="utf-8") as f:
        f.writelines(l for l in lines if json.loads(l)["arxiv_id"] in chon)
GIOI_HAN = f"--limit {SO_BAI}" if SO_BAI else ""   # also when the data came from the archives
print(f"Dữ liệu ở {DATA}: {len(os.listdir(f'{DATA}/corpus'))} bài, {time.time() - t0:.0f} s")

## Bước 3. Quét gatherer trên mọi phiên bản
Mỗi dòng của `logs/gather_survey.csv` là một phiên bản. Đáng xem: `several_roots` (nhiều file có `\documentclass`),
`guessed_root` (không có `\documentclass`), `with_missing_inputs` (`\input` tới file không có), `bbl_only` (tài liệu tham khảo
chỉ nằm trong file `.bbl`, mà Milestone 1 không giữ), `case_differs`, `cycle`.

In [ ]:
!python src/gatherer.py --survey --data-dir "{DATA}" {GIOI_HAN}
import pandas as pd
s = pd.read_csv(f"{DATA}/logs/gather_survey.csv", dtype={"problems": str}).fillna("")
print(s.status.value_counts().to_dict(), "| trung bình", round(s.files.mean(), 1), "file/phiên bản")
co_van_de = s[s.problems != ""]
print(f"{len(co_van_de)} phiên bản có ghi chú; vài ví dụ:")
display(co_van_de[["arxiv_id", "version", "root", "files", "missing", "problems"]].head(20))

## Bước 4. Chạy pipeline Milestone 2
In ra phần nào đang dùng bản tối giản (`stand-in`). Kết quả: `corpus/<id>/parsed/*.json` và `tables/` (Parquet).

In [ ]:
!python src/benchmark.py --data-dir "{DATA}" --interval 2 --disk-interval 120 -- python src/pipeline_m2.py --data-dir "{DATA}" --workers 2 {GIOI_HAN}
log = pd.read_csv(f"{DATA}/logs/parse_log.csv")
print(log.status.value_counts().to_dict(), f"| {log.seconds.sum():.0f} s parse, {log.seconds.mean():.3f} s/bài")
display(log.sort_values("seconds", ascending=False).head(5)[["arxiv_id", "versions", "elements", "edges", "seconds"]])
# Các dạng ghi chú trên toàn bộ phiên bản: gửi bảng này để xem code gặp gì trên dữ liệu thật
from collections import Counter
import re
dang = Counter()
for t in glob.glob(f"{DATA}/corpus/*/parsed/gathering.json"):
    for v in json.load(open(t, encoding="utf-8")).values():
        dang.update({re.sub(r"^[^ ]*\.tex: ", "", p)[:40] for p in v.get("problems", [])})
print(*[f"{n:6d}  {k}" for k, n in dang.most_common(30)], sep="\n")

## Bước 5. Xem cây của một bài

In [ ]:
from collections import defaultdict
BAI = None   # ví dụ "2605-12338"; None = bài đầu tiên có nhiều file
if BAI is None:
    nhieu_file = s[(s.status == "ok") & (s.files > 1)]
    BAI = (nhieu_file if len(nhieu_file) else s[s.status == "ok"]).arxiv_id.iloc[0]
p = f"{DATA}/corpus/{BAI}/parsed"
el = {r[0]: (r[1], r[2]) for r in json.load(open(f"{p}/elements.json", encoding="utf-8"))["rows"]}
edges = json.load(open(f"{p}/hierarchy.json", encoding="utf-8"))["rows"]
version = edges[0][0] if edges else 1
con = defaultdict(list)
for v, c, par in edges:
    if v == version:
        con[par].append(c)
root = next(e for e, (k, t) in el.items() if k == "document" and t == f"Version {version} root")
def in_cay(node, d=0, dong=[0]):
    for c in con[node]:
        if dong[0] >= 120:
            return
        k, t = el[c]
        print("  " * d + f"- {k}: {t[:100]}")
        dong[0] += 1
        in_cay(c, d + 1)
print(BAI, f"v{version}:", json.load(open(f"{p}/gathering.json"))[f"v{version}"]["files"])
in_cay(root)

## (Tuỳ chọn) Lưu kết quả quét lên Drive để gửi nhóm
Chỉ ghi 2 file CSV nhỏ vào `04_report/m2_thu/` trên Drive.

In [ ]:
DICH = "/content/drive/MyDrive/group11-data/04_report/m2_thu"
os.makedirs(DICH, exist_ok=True)
for ten in ("gather_survey.csv", "parse_log.csv"):
    shutil.copy2(f"{DATA}/logs/{ten}", f"{DICH}/{ten}")
print("Đã chép vào", DICH)